# Django, Forms

## Introduction to Django Forms
---

In this lesson, you'll learn how to **handle user input with Django Forms**.

Forms are essential for any web application that needs to collect data from users - whether it's a login page, a contact form, or a product order.

<br>

🔑 Think of Django Forms as a "smart intermediary" between your website and the user.

Their primary job is to ensure that the data people type into form fields reaches the database intact, securely, and in the correct format.

1. [Why Use Django Forms](#Why-Use-Django-Forms),
    - [The Problem with Raw HTML Forms](#The-Problem-with-Raw-HTML-Forms),
    - [What Django Forms Provide](#What-Django-Forms-Provide),
2. [Creating Your First Form](#Creating-Your-First-Form),
    - [Form Class and Fields](#Form-Class-and-Fields),
    - [Common Field Types](#Common-Field-Types),
3. [Rendering Forms in Templates](#Rendering-Forms-in-Templates),
    - [Quick Rendering Methods](#Quick-Rendering-Methods),
    - [Manual Field Rendering](#Manual-Field-Rendering),
    - [CSRF Protection](#CSRF-Protection),
4. [Processing Form Data](#Processing-Form-Data),
    - [The View Pattern](#The-View-Pattern),
    - [is_valid() and cleaned_data](#is_valid()-and-cleaned_data),
5. [Basic Field Validation](#Basic-Field-Validation),
    - [Built-in Validators](#Built-in-Validators),
    - [Field Arguments](#Field-Arguments),
6. [🧠 EXERCISE 🧠, Build Blog Search Form Flow](#🧠-EXERCISE-🧠,-Build-Blog-Search-Form-Flow),
7. [🧠 EXERCISE 🧠, Validate Existing blog_list_create POST](#🧠-EXERCISE-🧠,-Validate-Existing-blog_list_create-POST).

<br>

## Why Use Django Forms

---

### The Problem with Raw HTML Forms

---

You could handle forms with plain HTML and manually process `request.POST`:

```html
<form method="post">
    <input type="text" name="email">
    <input type="submit" value="Subscribe">
</form>
```

<br>

But this approach has **serious problems**:

| Problem | Risk |
|---------|------|
| No automatic validation | Invalid data gets into your database |
| No type conversion | Everything is a string |
| Manual error handling | Inconsistent user experience |
| Security vulnerabilities | XSS, CSRF attacks |
| Repetitive code | You write the same logic repeatedly |

<br>

### What Django Forms Provide

---

Django Forms solve all of these problems:

<br>

| Feature | Benefit |
|---------|--------|
| **Automatic HTML generation** | Consistent, accessible markup |
| **Data validation** | Invalid data is rejected before processing |
| **Type conversion** | Strings become proper Python types |
| **Error handling** | User-friendly error messages |
| **CSRF protection** | Built-in security against attacks |
| **Reusability** | Define once, use everywhere |

In Django's MVT (Model-View-Template) architecture, forms are a sort of "omnipresent bonus." They aren’t strictly hard-wired into any single one of those three layers; instead, they function as a bridge that connects them all.

<br>

## Creating Your First Form

---

### Form Class and Fields

---

Django forms are defined as Python classes that inherit from `forms.Form`.

Create a new file `forms.py` in your app directory:

In [ ]:
# blog/forms.py

from django import forms


class CommentForm(forms.Form):
    """Simple form aligned with current Comment model."""

    content = forms.CharField(widget=forms.Textarea, min_length=3)

<br>

**Key concepts:**

- Each **field** corresponds to an input element in the HTML form
- Field types (like `CharField`, `EmailField`) determine validation and HTML widget
- The **widget** controls how the field appears (text input, textarea, checkbox, etc.)

<br>

### Common Field Types

---

Django provides many built-in field types for different data:

<br>

| Field Type | HTML Widget | Python Type | Use For |
|------------|------------|-------------|--------|
| `CharField` | `<input type="text">` | `str` | Short text |
| `EmailField` | `<input type="email">` | `str` | Email addresses |
| `IntegerField` | `<input type="number">` | `int` | Whole numbers |
| `DecimalField` | `<input type="number">` | `Decimal` | Money, precise numbers |
| `BooleanField` | `<input type="checkbox">` | `bool` | Yes/No choices |
| `DateField` | `<input type="text">` | `date` | Dates |
| `ChoiceField` | `<select>` | `str` | Dropdown selection |
| `FileField` | `<input type="file">` | `UploadedFile` | File uploads |

In [ ]:
# Example: A more complex form with various field types

from django import forms
from blog.models import CategoryType

class BlogReviewForm(forms.Form):
    """Form aligned with BlogReview model and Blog categories."""

    title = forms.CharField(max_length=200)
    author_email = forms.EmailField(required=False)
    rating = forms.IntegerField(min_value=1, max_value=5)
    category = forms.ChoiceField(choices=CategoryType.choices)
    publish_now = forms.BooleanField(required=False)  # Optional checkbox
    review_comment = forms.CharField(
        widget=forms.Textarea,
        required=False  # Optional field
    )

<br>

## Rendering Forms in Templates

---

### Quick Rendering Methods

---

Django provides several shortcuts to render the entire form at once:

<br>

| Method | Output Format |
|--------|---------------|
| `{{ form.as_p }}` | Each field wrapped in `<p>` tags |
| `{{ form.as_table }}` | Each field in a `<tr>` (requires `<table>`) |
| `{{ form.as_ul }}` | Each field in an `<li>` (requires `<ul>`) |
| `{{ form.as_div }}` | Each field wrapped in `<div>` tags (Django 4.0+) |

In [ ]:
# Example template: templates/blog/comment_form.html

template_example = """
<!DOCTYPE html>
<html>
<head>
    <title>Add Comment</title>
</head>
<body>
    <h1>Add Comment</h1>

    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">Save Comment</button>
    </form>
</body>
</html>
"""
print(template_example)

<br>

**Important:** Always include `{% csrf_token %}` inside your form! This is required for security.

<br>

🔑 Briefly:

- The user's browser holds a login session cookie for your website.

- An attacker lures the user to a third-party site that secretly sends a POST request to your endpoint.

- Without protection, the server would see a valid cookie and execute the action (e.g., post a comment, change a profile...).

In [ ]:
# Example template: templates/blog/comment_success.html

template_success = """
<!DOCTYPE html>
<html>
<head>
    <title>Comment Success</title>
"""

In [ ]:
# blog/views.py

def comment_form_view(request):
    if request.method == "POST":
        form = BlogReviewForm(request.POST)
        if form.is_valid():
            # This lesson form is a plain forms.Form, not a ModelForm.
            # We only validate input here and then redirect (PRG pattern).
            return redirect("blog:comment-form-success")
    else:
        form = BlogReviewForm()
    return render(request, "blog/comment_form.html", {"form": form})


def comment_form_success(request):
    return render(request, "blog/comment_success.html")

In [ ]:
# blog/urls.py

urlpatterns = [
    path("comment-form/", views.comment_form_view, name="comment-form"),
    path("comment-form/success/", views.comment_form_success, name="comment-form-success"),
]


<br>

### Manual Field Rendering

---

For more control over the HTML output, you can render fields individually:

In [ ]:
# Manual field rendering example

template_manual = """
<form method="post">
    {% csrf_token %}
    
    <div class="form-group">
        <label for="{{ form.name.id_for_label }}">Your Name:</label>
        {{ form.name }}
        {% if form.name.errors %}
            <span class="error">{{ form.name.errors.0 }}</span>
        {% endif %}
    </div>
    
    <div class="form-group">
        <label for="{{ form.email.id_for_label }}">Email:</label>
        {{ form.email }}
        {% if form.email.errors %}
            <span class="error">{{ form.email.errors.0 }}</span>
        {% endif %}
    </div>
    
    <div class="form-group">
        <label for="{{ form.message.id_for_label }}">Message:</label>
        {{ form.message }}
    </div>
    
    <button type="submit">Send</button>
</form>
"""
print(template_manual)

<br>

**Available attributes for each field:**

| Attribute | Description |
|-----------|-------------|
| `{{ form.fieldname }}` | The input widget itself |
| `{{ form.fieldname.label }}` | The field's label text |
| `{{ form.fieldname.id_for_label }}` | The HTML id attribute |
| `{{ form.fieldname.errors }}` | List of validation errors |
| `{{ form.fieldname.help_text }}` | Help text if defined |
| `{{ form.fieldname.value }}` | The current value |

<br>

### CSRF Protection

---

**CSRF** (Cross-Site Request Forgery) is an attack where a malicious site tricks a user's browser into submitting a form to your site.

<br>

Django protects against this with a token:

1. When rendering the form, Django generates a unique token
2. The token is included as a hidden field in the form
3. When the form is submitted, Django verifies the token matches

<br>

**Always include the CSRF token in POST forms:**

```html
<form method="post">
    {% csrf_token %}  <!-- This is REQUIRED -->
    {{ form.as_p }}
    <button type="submit">Submit</button>
</form>
```

Without `{% csrf_token %}`, Django will reject the form submission with a 403 Forbidden error.

<br>

## Processing Form Data

---

### The View Pattern

---

The standard pattern for handling forms in Django views:

In [ ]:
# blog/views.py

import json
from django.http import JsonResponse
from django.views.decorators.csrf import csrf_exempt

from blog.models import Blog


@csrf_exempt  # Demo-only: use CSRF token in production templates
def blog_create(request):
    """Simple GET/POST flow used in this project."""
    if request.method != 'POST':
        return JsonResponse({'tip': 'Send POST with title, author, published_date'})

    data = json.loads(request.body)
    blog = Blog.objects.create(
        title=data['title'],
        author=data['author'],
        published_date=data['published_date'],
    )
    return JsonResponse({'id': blog.id, 'title': blog.title}, status=201)

<br>

**The flow explained:**

1. **GET request**: User visits the page → show empty form
2. **POST request**: User submits the form → validate and process
3. **Valid data**: Process data and redirect (PRG pattern)
4. **Invalid data**: Show form again with errors

<br>

**PRG Pattern** (Post-Redirect-Get): After successful form submission, always redirect. This prevents duplicate submissions when the user refreshes the page.

<br>

### `is_valid()` and `cleaned_data`

---

Two key concepts when processing form data:

<br>

**`is_valid()`** - Validates all form fields:
- Returns `True` if all fields pass validation
- Returns `False` if any field has errors
- Populates `form.errors` with error messages

In [ ]:
# Example: Checking form validity

from django import forms


class SimpleForm(forms.Form):
    email = forms.EmailField()
    age = forms.IntegerField(min_value=0)


# Simulate form with invalid data
form = SimpleForm(data={'email': 'not-an-email', 'age': '-5'})

print(f"Is valid: {form.is_valid()}")
print(f"Errors: {form.errors}")

<br>

**`cleaned_data`** - Dictionary of validated and converted data:
- Only available after calling `is_valid()` and getting `True`
- Contains Python objects, not raw strings
- Missing optional fields have their default values

In [ ]:
# Example: Accessing cleaned_data

form = SimpleForm(data={'email': 'user@example.com', 'age': '25'})

if form.is_valid():
    print(f"Email: {form.cleaned_data['email']}")
    print(f"Age: {form.cleaned_data['age']}")
    print(f"Age type: {type(form.cleaned_data['age'])}")

<br>

**Important:** Always access user data through `cleaned_data`, never directly from `request.POST`. The `cleaned_data` values are:
- Validated
- Converted to proper Python types
- Sanitized

<br>

## Basic Field Validation

---

### Built-in Validators

---

🔑 Validators are small, portable snippets of code that you attach to form fields to prevent users from sending nonsense to your database.


Django provides many built-in validators that you can attach to fields:

In [ ]:
# blog/forms.py
from django import forms
from blog.models import CategoryType
from django.core.validators import MinValueValidator, MaxValueValidator, RegexValidator


class BlogCreateForm(forms.Form):
    title = forms.CharField(max_length=200)
    author = forms.CharField(max_length=100, validators=[
            RegexValidator(
                regex=r'^[a-zA-Z0-9_]+$',
                message='Username can only contain letters, numbers, and underscores.'
            )])
    author_email = forms.EmailField(required=False)
    published_date = forms.DateField()
    category_type = forms.ChoiceField(choices=CategoryType.choices)


class CommentCreateForm(forms.Form):
    content = forms.CharField(
        min_length=3,
        widget=forms.Textarea,
    )


class BlogReviewForm(forms.Form):
    rating = forms.IntegerField(
        validators=[MinValueValidator(1), MaxValueValidator(5)]
    )
    comment = forms.CharField(required=False, widget=forms.Textarea)

### Quick shell check (render + validation)

Use Django shell to verify both HTML rendering and validation errors:

```bash
from blog.forms import BlogCreateForm, CommentCreateForm, BlogReviewForm

print(BlogCreateForm().as_p())  # visual check of generated fields/widgets

invalid_blog = BlogCreateForm(data={
    "title": "",
    "author": "bad user!",
    "author_email": "not-an-email",
    "published_date": "2026-99-99",
    "category_type": "invalid",
})
invalid_blog.is_valid()
invalid_blog.errors.as_json()

print("\n=== Invalid CommentCreateForm ===")
invalid_comment = CommentCreateForm(data={"content": "ok"})
print("is_valid:", invalid_comment.is_valid())
print(invalid_comment.errors.as_json())

print("\n=== Invalid BlogReviewForm ===")
invalid_review = BlogReviewForm(data={"rating": 7, "comment": "too high"})
print("is_valid:", invalid_review.is_valid())
print(invalid_review.errors.as_json())
PY
```

Best practice: prefer `errors.as_json()` in shell because it is deterministic and easier to compare than pretty-printed HTML errors.

<br>

| Validator | Purpose | Example |
|-----------|---------|--------|
| `MinLengthValidator` | Minimum string length | `MinLengthValidator(3)` |
| `MaxLengthValidator` | Maximum string length | `MaxLengthValidator(100)` |
| `MinValueValidator` | Minimum numeric value | `MinValueValidator(0)` |
| `MaxValueValidator` | Maximum numeric value | `MaxValueValidator(100)` |
| `RegexValidator` | Match a pattern | `RegexValidator(r'^[A-Z]+$')` |
| `EmailValidator` | Valid email (automatic in EmailField) | Built into EmailField |
| `URLValidator` | Valid URL (automatic in URLField) | Built into URLField |

<br>

### Field Arguments

---

Every field accepts common arguments for validation and display:

In [ ]:
from django import forms


class DetailedForm(forms.Form):
    """Form demonstrating common field arguments."""
    
    # required (default=True) - field must have a value
    required_field = forms.CharField(required=True)  # Default
    optional_field = forms.CharField(required=False)
    
    # initial - pre-filled value when form is displayed
    country = forms.CharField(initial='United States')
    
    # label - custom label text (default is field name capitalized)
    email_address = forms.EmailField(label='Your Email')
    
    # help_text - additional guidance for the user
    password = forms.CharField(
        widget=forms.PasswordInput,
        help_text='Must be at least 8 characters.'
    )
    
    # error_messages - customize error messages
    age = forms.IntegerField(
        min_value=0,
        error_messages={
            'required': 'Please enter your age.',
            'min_value': 'Age cannot be negative.',
            'invalid': 'Please enter a valid number.'
        }
    )

<br>

| Argument | Purpose | Example |
|----------|---------|--------|
| `required` | Whether field must have a value | `required=False` |
| `initial` | Default value for display | `initial='US'` |
| `label` | Custom label text | `label='Your Name'` |
| `help_text` | Guidance text | `help_text='Enter your email'` |
| `widget` | HTML input type | `widget=forms.Textarea` |
| `error_messages` | Custom error messages | `{'required': 'This is required'}` |
| `validators` | List of validator functions | `validators=[my_validator]` |

<br>

## Complete Example: Blog Search Form

---

Let's put it all together with a complete working example aligned to current `blog` app (`Blog` fields: `title`, `author`, `published_date`, `category_type`):

In [ ]:
# blog/forms.py

from django import forms


class BlogSearchForm(forms.Form):
    """Form aligned with Blog fields used in current views.py."""

    SORT_CHOICES = [
        ('title', 'Title'),
        ('author', 'Author'),
        ('published_date', 'Published date'),
    ]

    q = forms.CharField(
        required=False,
        label='Search by title',
        help_text='Case-insensitive search over Blog.title',
    )
    sort = forms.ChoiceField(
        choices=SORT_CHOICES,
        initial='title',
        label='Sort by',
    )

In [ ]:
# blog/views.py

from django.http import JsonResponse, HttpRequest

from .forms import BlogSearchForm
from blog.models import Blog


def blog_search(request: HttpRequest) -> JsonResponse:
    form = BlogSearchForm(request.GET)
    if not form.is_valid():
        return JsonResponse({'errors': form.errors}, status=400)
    query = form.cleaned_data['q']
    sort = form.cleaned_data['sort']
    blogs = Blog.objects.all()
    if query:
        blogs = blogs.filter(title__icontains=query)
    blogs = blogs.order_by(sort)  # safe: sort je omezen ChoiceField
    return JsonResponse(list(blogs.values()), safe=False)

In [ ]:
# templates/blog/search.html

template_complete = """
{% extends 'blog/base.html' %}

{% block title %}Blog Search{% endblock %}

{% block content %}
<h1>Search Blogs</h1>

<form method="get">
    <label for="q">Title contains</label>
    <input type="text" id="q" name="q" value="{{ form.q.value|default:'' }}">

    <label for="sort">Sort</label>
    <select id="sort" name="sort">
        <option value="title" {% if form.sort.value == 'title' %}selected{% endif %}>Title</option>
        <option value="author" {% if form.sort.value == 'author' %}selected{% endif %}>Author</option>
        <option value="published_date" {% if form.sort.value == 'published_date' %}selected{% endif %}>Published date</option>
    </select>

    <button type="submit">Search</button>
</form>

<hr>
<h2>Results ({{ blogs|length }})</h2>
{% if blogs %}
<ul>
    {% for blog in blogs %}
    <li>
        <strong>{{ blog.title }}</strong> - {{ blog.author }} ({{ blog.published_date }})
    </li>
    {% endfor %}
</ul>
{% else %}
<p>No blogs found for current query.</p>
{% endif %}
{% endblock %}
"""
print(template_complete)

### Quick manual test inputs (browser + curl)

Use these concrete values to verify the GET search flow quickly.

**Browser inputs (form):**
- `Title contains (q)`: `django`
- `Sort`: `title`
- Click **Search**

Expected URL after submit:
- `http://127.0.0.1:8000/blog/search/?q=django&sort=title`

Try also these cases:
- `q=python`, `sort=published_date`
- `q=` (empty), `sort=author`

**curl checks (same query params):**

```bash
# valid query
curl "http://127.0.0.1:8000/blog/search/?q=django&sort=title"

# valid: empty q, allowed sort
curl "http://127.0.0.1:8000/blog/search/?q=&sort=author"

# invalid sort (should be rejected if form validation is wired)
curl -i "http://127.0.0.1:8000/blog/search/?q=django&sort=drop_table"
```

Best practice: keep `sort` restricted by `ChoiceField` and never pass it directly to `order_by()` without whitelist/form validation.

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Form class** | Inherit from `forms.Form`, define fields as class attributes |
| **Field types** | `CharField`, `EmailField`, `IntegerField`, `ChoiceField`, etc. |
| **Rendering** | `{{ form.as_p }}` for quick, manual for control |
| **CSRF token** | Always include `{% csrf_token %}` in POST forms |
| **View pattern** | Check method → bind data → validate → process or re-display |
| **is_valid()** | Returns `True` if all fields pass validation |
| **cleaned_data** | Dictionary of validated, type-converted data |
| **Validators** | Built-in or custom functions to validate field values |
| **PRG pattern** | After success, redirect to prevent duplicate submissions |

<br>

### 🧠 EXERCISE 🧠, Build Blog Search Form Flow

---

Create a blog-focused form flow for the current project:

1. Create `blog/forms.py` with a `BlogSearchForm` class containing:
   - `q` (CharField, optional)
   - `sort` (ChoiceField with values: `title`, `author`, `published_date`)

2. Use/adjust the existing `blog_search` view in `blog/views.py` so that:
   - GET parameters are validated through the form
   - Search is done using `Blog.title`
   - Sorting is whitelisted to real `Blog` fields

3. Create template `templates/blog/search.html` for GET search form rendering

4. Add URL pattern in `blog/urls.py`

5. Test with valid and invalid query params

<details>
    <summary>▶️ Solution</summary>
    
**1. blog/forms.py:**

```python
from django import forms


class BlogSearchForm(forms.Form):
    SORT_CHOICES = [
        ('title', 'Title'),
        ('author', 'Author'),
        ('published_date', 'Published date'),
    ]

    q = forms.CharField(required=False)
    sort = forms.ChoiceField(choices=SORT_CHOICES, initial='title')
```

**2. blog/views.py (form + existing query logic):**

```python
from django.http import JsonResponse, HttpRequest
from .forms import BlogSearchForm
from .models import Blog


def blog_search(request: HttpRequest) -> JsonResponse:
    form = BlogSearchForm(request.GET)
    if not form.is_valid():
        return JsonResponse({'errors': form.errors}, status=400)

    query = form.cleaned_data['q']
    sort = form.cleaned_data['sort']

    blogs = Blog.objects.all()
    if query:
        blogs = blogs.filter(title__icontains=query)

    # sort is already constrained by form ChoiceField
    blogs = blogs.order_by(sort)

    return JsonResponse(list(blogs.values()), safe=False)
```

**3. templates/blog/search.html:**

```html
<form method="get">
    {{ form.as_p }}
    <button type="submit">Search</button>
</form>
```

**4. blog/urls.py:**

```python
from django.urls import path
from . import views

urlpatterns = [
    path('search/', views.blog_search, name='blog_search'),
]
```

**5. Test:**
- Run `python manage.py runserver`
- Visit `http://127.0.0.1:8000/blog/search/?q=django&sort=title`
- Try invalid `sort` and verify the form-driven validation response
</details>

<br>

### 🧠 EXERCISE 🧠, Validate Existing blog_list_create POST

---

Improve the current API-style create endpoint:

1. Create `BlogCreateForm` with:
   - `title` (CharField, max 200)
   - `author` (CharField, max 100)
   - `published_date` (DateField)

2. Update existing `blog_list_create` in `blog/views.py` so that POST:
   - Validates incoming JSON through `BlogCreateForm`
   - Returns form errors with status `400` when invalid
   - Creates `Blog` row from `cleaned_data` when valid

3. Keep GET branch unchanged (`count` + `results`)

**Hint:** Parse JSON first, then instantiate `BlogCreateForm(data)`

<details>
    <summary>▶️ Solution</summary>

**blog/forms.py:**

```python
from django import forms


class BlogCreateForm(forms.Form):
    title = forms.CharField(max_length=200)
    author = forms.CharField(max_length=100)
    published_date = forms.DateField()
```

**blog/views.py (existing view improved with form validation):**

```python
import json
from django.http import JsonResponse, HttpRequest
from django.views.decorators.csrf import csrf_exempt
from django.views.decorators.http import require_http_methods

from .forms import BlogCreateForm
from .models import Blog


@csrf_exempt
@require_http_methods(["GET", "POST"])
def blog_list_create(request: HttpRequest) -> JsonResponse:
    if request.method == 'GET':
        blogs = list(
            Blog.objects.order_by('-id').values('id', 'title', 'author', 'published_date')
        )
        return JsonResponse({'count': len(blogs), 'results': blogs})

    data = json.loads(request.body)
    form = BlogCreateForm(data)
    if not form.is_valid():
        return JsonResponse({'errors': form.errors}, status=400)

    blog = Blog.objects.create(**form.cleaned_data)
    return JsonResponse({'id': blog.id, 'title': blog.title}, status=201)
```

**blog/urls.py:**

```python
from django.urls import path
from . import views

urlpatterns = [
    path('blogs/', views.blog_list_create, name='blog_list_create'),
]
```
</details>

---